# Classroom Engagement Detection (Notebook Version)

This notebook runs real-time classroom engagement detection from webcam or RTSP.

- Uses `yolov8n-pose.pt`
- No Flask server
- No identity tracking or face recognition
- Logs average engagement every 30 seconds to SQLite with snapshot images

Press `q` in the OpenCV window to stop the live loop.

In [2]:
import sqlite3
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import List, Optional, Tuple

import cv2
import numpy as np
from ultralytics import YOLO


# COCO keypoint indices for YOLOv8 pose
NOSE = 0
LEFT_EYE = 1
RIGHT_EYE = 2
LEFT_EAR = 3
RIGHT_EAR = 4
LEFT_SHOULDER = 5
RIGHT_SHOULDER = 6
LEFT_HIP = 11
RIGHT_HIP = 12

MIN_KPT_CONF = 0.35
STILL_MOTION_NORM = 0.008
ACTIVE_MOTION_NORM_MAX = 0.045
CHAOTIC_MOTION_NORM = 0.09
STILL_FRAME_LIMIT = 30


@dataclass
class MotionSlot:
    center: Tuple[float, float]
    still_count: int


class AnonymousMotionEstimator:
    """Anonymous short-lived motion slots, no identity persistence."""

    def __init__(self, max_match_px: float = 100.0) -> None:
        self.max_match_px = max_match_px
        self._prev_slots: List[MotionSlot] = []

    def update(self, centers: List[Tuple[float, float]]) -> List[Tuple[float, int]]:
        if not centers:
            self._prev_slots = []
            return []

        if not self._prev_slots:
            self._prev_slots = [MotionSlot(center=c, still_count=0) for c in centers]
            return [(0.0, 0) for _ in centers]

        used_prev = set()
        output: List[Tuple[float, int]] = []
        new_slots: List[MotionSlot] = []

        for center in centers:
            best_idx = None
            best_dist = float("inf")

            for idx, slot in enumerate(self._prev_slots):
                if idx in used_prev:
                    continue
                dist = float(np.linalg.norm(np.array(center) - np.array(slot.center)))
                if dist < best_dist and dist <= self.max_match_px:
                    best_dist = dist
                    best_idx = idx

            if best_idx is None:
                displacement = 0.0
                still_count = 0
            else:
                used_prev.add(best_idx)
                prev_still = self._prev_slots[best_idx].still_count
                displacement = best_dist
                still_count = prev_still + 1 if displacement < 2.0 else 0

            output.append((displacement, still_count))
            new_slots.append(MotionSlot(center=center, still_count=still_count))

        self._prev_slots = new_slots
        return output


class EngagementDBLogger:
    def __init__(self, db_path: Path, snapshot_dir: Path, period_id: str, interval_sec: int = 30) -> None:
        self.db_path = db_path
        self.snapshot_dir = snapshot_dir
        self.period_id = period_id
        self.interval_sec = interval_sec
        self._window: List[float] = []
        self._last_flush = time.time()

        self.snapshot_dir.mkdir(parents=True, exist_ok=True)
        self._init_db()

    def _init_db(self) -> None:
        conn = sqlite3.connect(self.db_path)
        try:
            conn.execute(
                """
                CREATE TABLE IF NOT EXISTS engagement_logs (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    period_id TEXT NOT NULL,
                    timestamp TEXT NOT NULL,
                    avg_engagement_percent REAL NOT NULL,
                    snapshot_path TEXT NOT NULL
                )
                """
            )
            conn.commit()
        finally:
            conn.close()

    def add_sample(self, engagement_percent: float, frame: np.ndarray) -> None:
        self._window.append(float(engagement_percent))
        now = time.time()
        if now - self._last_flush >= self.interval_sec:
            self._flush(frame)
            self._last_flush = now

    def flush_remaining(self, frame: Optional[np.ndarray]) -> None:
        if frame is None or not self._window:
            return
        self._flush(frame)

    def _flush(self, frame: np.ndarray) -> None:
        if not self._window:
            return

        avg_percent = float(np.mean(self._window))
        ts = datetime.utcnow().replace(microsecond=0)
        stamp = ts.strftime("%Y%m%d_%H%M%S")
        snapshot_file = self.snapshot_dir / f"{self.period_id}_{stamp}.jpg"
        cv2.imwrite(str(snapshot_file), frame)

        conn = sqlite3.connect(self.db_path)
        try:
            conn.execute(
                """
                INSERT INTO engagement_logs (period_id, timestamp, avg_engagement_percent, snapshot_path)
                VALUES (?, ?, ?, ?)
                """,
                (self.period_id, ts.isoformat(), round(avg_percent, 2), str(snapshot_file)),
            )
            conn.commit()
        finally:
            conn.close()

        self._window.clear()


class EngagementAnalyzer:
    def __init__(self, model_path: str = "yolov8n-pose.pt", conf_thres: float = 0.3, imgsz: int = 640) -> None:
        self.model = YOLO(model_path)
        self.conf_thres = conf_thres
        self.imgsz = imgsz
        self.motion = AnonymousMotionEstimator()

    @staticmethod
    def _visible_point(kpts_xy: np.ndarray, kpts_conf: np.ndarray, idx: int):
        if idx >= len(kpts_xy) or float(kpts_conf[idx]) < MIN_KPT_CONF:
            return None
        return kpts_xy[idx]

    @classmethod
    def _head_forward(cls, kpts_xy: np.ndarray, kpts_conf: np.ndarray) -> bool:
        nose = cls._visible_point(kpts_xy, kpts_conf, NOSE)
        left_eye = cls._visible_point(kpts_xy, kpts_conf, LEFT_EYE)
        right_eye = cls._visible_point(kpts_xy, kpts_conf, RIGHT_EYE)
        left_ear = cls._visible_point(kpts_xy, kpts_conf, LEFT_EAR)
        right_ear = cls._visible_point(kpts_xy, kpts_conf, RIGHT_EAR)

        if nose is None:
            return False

        if left_eye is None or right_eye is None:
            if left_ear is not None and right_ear is not None:
                return abs(float(right_ear[0] - left_ear[0])) > 20.0
            return False

        eye_center_x = float((left_eye[0] + right_eye[0]) / 2.0)
        eye_span = max(abs(float(right_eye[0] - left_eye[0])), 1.0)
        nose_offset_ratio = abs(float(nose[0] - eye_center_x)) / eye_span

        eye_line_y = float((left_eye[1] + right_eye[1]) / 2.0)
        downward_ratio = float(nose[1] - eye_line_y) / eye_span

        is_centered = nose_offset_ratio <= 0.55
        not_looking_down = downward_ratio <= 1.05

        if left_ear is not None and right_ear is not None:
            ear_balance = abs(float((nose[0] - left_ear[0]) - (right_ear[0] - nose[0])))
            ear_span = max(abs(float(right_ear[0] - left_ear[0])), 1.0)
            return is_centered and not_looking_down and (ear_balance / ear_span) <= 0.7

        return is_centered and not_looking_down

    @classmethod
    def _torso_center(cls, kpts_xy: np.ndarray, kpts_conf: np.ndarray):
        idxs = [LEFT_SHOULDER, RIGHT_SHOULDER, LEFT_HIP, RIGHT_HIP]
        valid = []
        for idx in idxs:
            pt = cls._visible_point(kpts_xy, kpts_conf, idx)
            if pt is not None:
                valid.append(pt)

        if len(valid) < 2:
            return None

        arr = np.array(valid)
        return float(arr[:, 0].mean()), float(arr[:, 1].mean())

    @staticmethod
    def _classify_activity(motion_norm: float, still_count: int) -> str:
        if motion_norm >= CHAOTIC_MOTION_NORM:
            return "chaotic"
        if motion_norm < STILL_MOTION_NORM:
            return "still_disengaged" if still_count >= STILL_FRAME_LIMIT else "still"
        if motion_norm <= ACTIVE_MOTION_NORM_MAX:
            return "active"
        return "restless"

    @staticmethod
    def score_label(engaged_ratio: float):
        if engaged_ratio >= 0.7:
            return "High", (0, 200, 0)
        if engaged_ratio >= 0.4:
            return "Medium", (0, 215, 255)
        return "Low", (0, 0, 255)

    def analyze(self, frame: np.ndarray):
        annotated = frame.copy()
        result = self.model.predict(
            source=frame,
            conf=self.conf_thres,
            imgsz=self.imgsz,
            classes=[0],
            verbose=False,
        )[0]

        if result.boxes is None or len(result.boxes) == 0 or result.keypoints is None:
            self.motion.update([])
            metrics = {
                "timestamp": datetime.utcnow().replace(microsecond=0).isoformat(),
                "total_persons": 0,
                "engaged": 0,
                "distracted": 0,
                "engagement_score": "Low",
                "engagement_percent": 0,
            }
            cv2.putText(annotated, "Engagement: Low (0%)", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
            return annotated, metrics

        boxes = result.boxes.xyxy.cpu().numpy()
        keypoints_xy = result.keypoints.xy.cpu().numpy()
        keypoints_conf = result.keypoints.conf.cpu().numpy()

        torso_centers: List[Tuple[float, float]] = []
        box_diags: List[float] = []
        head_forward_list: List[bool] = []

        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = [int(v) for v in box]
            kxy = keypoints_xy[i]
            kcf = keypoints_conf[i]

            head_forward_list.append(self._head_forward(kxy, kcf))
            center = self._torso_center(kxy, kcf)
            if center is None:
                center = ((x1 + x2) / 2.0, (y1 + y2) / 2.0)

            torso_centers.append(center)
            box_diags.append(max(float(np.hypot(x2 - x1, y2 - y1)), 1.0))

        motion_data = self.motion.update(torso_centers)

        engaged_count = 0
        distracted_count = 0

        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = [int(v) for v in box]
            motion_px, still_count = motion_data[i]
            motion_norm = motion_px / box_diags[i]
            activity = self._classify_activity(motion_norm, still_count)

            engaged = head_forward_list[i] and activity in {"active", "still"}
            if activity in {"chaotic", "restless", "still_disengaged"}:
                engaged = False

            if engaged:
                engaged_count += 1
                color = (0, 200, 0)
                label = "Engaged ✅"
            else:
                distracted_count += 1
                color = (0, 0, 255)
                label = "Distracted ❌"

            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
            cv2.putText(annotated, label, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        total_persons = len(boxes)
        engaged_ratio = (engaged_count / total_persons) if total_persons else 0.0
        engagement_percent = int(round(engaged_ratio * 100))
        score, score_color = self.score_label(engaged_ratio)

        overlay = f"Engagement: {score} ({engagement_percent}%) | Persons: {total_persons}"
        cv2.putText(annotated, overlay, (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0, score_color, 2)

        metrics = {
            "timestamp": datetime.utcnow().replace(microsecond=0).isoformat(),
            "total_persons": total_persons,
            "engaged": engaged_count,
            "distracted": distracted_count,
            "engagement_score": score,
            "engagement_percent": engagement_percent,
        }
        return annotated, metrics

In [3]:
# ---- Configuration ----

SOURCE = 0  # Webcam index; for RTSP use: "rtsp://username:password@ip:554/stream"
MODEL_PATH = "yolov8n-pose.pt"
CONF = 0.3
IMGSZ = 640
PERIOD_ID = datetime.utcnow().strftime("period_%Y%m%d_%H%M%S")
DB_PATH = Path("engagement_logs.db")
SNAPSHOT_DIR = Path("snapshots")
MAX_FPS = 0  # 0 means uncapped

analyzer = EngagementAnalyzer(model_path=MODEL_PATH, conf_thres=CONF, imgsz=IMGSZ)
logger = EngagementDBLogger(db_path=DB_PATH, snapshot_dir=SNAPSHOT_DIR, period_id=PERIOD_ID)

print("Ready.")
print("Press 'q' in the video window to stop.")

Ready.
Press 'q' in the video window to stop.


In [4]:
# ---- Run real-time engagement detection ----

cap = cv2.VideoCapture(SOURCE)
if not cap.isOpened():
    raise RuntimeError(f"Unable to open video source: {SOURCE}")

cap.set(cv2.CAP_PROP_BUFFERSIZE, 2)

last_frame = None
last_metrics = None

try:
    while True:
        loop_start = time.time()
        ok, frame = cap.read()
        if not ok:
            print("Frame read failed; retrying...")
            time.sleep(0.05)
            continue

        annotated, metrics = analyzer.analyze(frame)
        logger.add_sample(metrics["engagement_percent"], annotated)

        last_frame = annotated
        last_metrics = metrics

        cv2.imshow("Classroom Engagement", annotated)

        # Press q to stop.
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

        if MAX_FPS > 0:
            elapsed = time.time() - loop_start
            target = 1.0 / MAX_FPS
            if elapsed < target:
                time.sleep(target - elapsed)
finally:
    cap.release()
    cv2.destroyAllWindows()
    logger.flush_remaining(last_frame)

if last_metrics is not None:
    print("Final aggregate (latest frame):")
    print(last_metrics)
    # Example payload shape:
    # {
    #   "timestamp": "2026-03-26T10:42:00",
    #   "total_persons": 28,
    #   "engaged": 21,
    #   "distracted": 7,
    #   "engagement_score": "High",
    #   "engagement_percent": 75
    # }

Final aggregate (latest frame):
{'timestamp': '2026-03-26T09:48:07', 'total_persons': 2, 'engaged': 1, 'distracted': 1, 'engagement_score': 'Medium', 'engagement_percent': 50}
